In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp 

# import qutip as qq

In [ ]:
# Helper Function

# Easier using 2->3


# Convertion helper 
# 1-> wave function  
# 2-> theta,phi 
# 3-> n,n location for hamiltonian
def C12(k,size):
    x=(k)%size[0]
    y=(k)//size[0]
    return x,y

def C13(k,j,size):
    i=k
    return i,j

def C21(pos,size):
    k=pos[0]+pos[1]*size[0]
    return k

def C23(pos,j,size):
    i=C21(pos,size)
    return i,j

def C31(i,j=0):
    return i

def C32(pos,size):
    x,y=C12(pos[0],size)
    return x,y

# Reconstruct Helper
def R12(par,size):
    i=np.array(range(size[0]*size[1]))
    val=np.zeros([size[1],size[0]],dtype=complex)
    x,y=C12(i,size)
    val[y,x]=par[i]
    return val

def R21(par,size):
    ipos=np.zeros([2,size[0]*size[1]],dtype=int)
    val=np.zeros(size[0]*size[1],dtype=complex)
    for j in range(size[1]):
        for i in range(size[0]):
            ipos[0,j*size[0]+i]=i
            ipos[1,j*size[0]+i]=j
    k=C21(ipos,size)
    r=range(size[0]*size[1])
    val[k]=par[ipos[0,r],ipos[1,r]]
    return val


def FDiffTheta(H,pos,a,dt,j,size):
    b=a/(2*dt)

    if pos[1]!=0:
        H[j,C21(pos+[0,-1],size)]-=b
    else:
        H[j,C21([pos[0],size[1]-1],size)]-=b
    if pos[1]!=(size[1]-1):
        H[j,C21(pos+[0, 1],size)]+=b
    else:
        H[j,C21([pos[0],0],size)]+=b
    return H

def SDiffTheta(H,pos,a,dt,j,size):
    b=a/dt**2

    H[j,C21(pos,size)]-=2*b
    if pos[1]!=0:
        H[j,C21(pos+[0,-1],size)]+=b
    else:
        H[j,C21([pos[0],size[1]-1],size)]+=b
    if pos[1]!=(size[1]-1):
        H[j,C21(pos+[0, 1],size)]+=b
    else:
        H[j,C21([pos[0],0],size)]+=b
    return H

def SDiffPhi(H,pos,a,dt,j,size):
    b=a/dt**2

    H[j,C21(pos,size)]-=2*b
    if pos[0]!=0:
        H[j,C21(pos+[-1,0],size)]+=b
    if pos[0]!=(size[0]-1):
        H[j,C21(pos+[ 1,0],size)]+=b
    return H

In [ ]:
# Functionized Main Varies alot of stuff
def maingeneral(phiE,ng,size,sphi=4*np.pi,stheta=np.pi,Isincludestate=False,GetHamiltonian=False,base=1,tpRatio=1,alpha=1,beta=1):
    N,M=size
    NM=N*M

    H=np.zeros([NM,NM],dtype=complex)
    phi=np.linspace(-sphi,sphi,N)
    theta=np.linspace(-stheta,stheta,M)
    dp=phi[1]-phi[0]
    dt=theta[1]-theta[0]

    GHz=1000000000

    Ecp=base*GHz
    Ect=tpRatio*base*GHz
    El=alpha*GHz
    Ej=beta*GHz

    count=0
    for j in range(M):
        for i in range(N):
            pos=np.array([i,j])
            H[count,C21(pos,size)]= +4*(Ect*ng**2)\
                                    +El*(phi[i]-np.pi*phiE)**2\
                                    -2*Ej*np.cos(phi[i])*np.cos(theta[j])
            H=SDiffPhi(H,pos,-4*Ecp,dp,count,size)
            H=FDiffTheta(H,pos,+1j*8*Ect*ng,dt,count,size)
            H=SDiffTheta(H,pos,-4*Ect,dt,count,size)
            count+=1
    if GetHamiltonian:
        return H
    if Isincludestate:
        a,b=np.linalg.eigh(H)
        n=np.argsort(a.real)
        asort=a[n]
        wf=b[:,n]
        return asort,wf
    a=np.linalg.eigvalsh(H)
    n=np.argsort(a.real)
    asort=a[n]
    return asort


In [ ]:
# Functionized Main Varies alot of stuff
def maingeneralsph(phiE,ng,size,sphi=4*np.pi,stheta=np.pi,Isincludestate=False,GetHamiltonian=False,base=1,tpRatio=1,alpha=1,beta=1,Enum=6):
    N,M=size
    NM=N*M

    H=np.zeros([NM,NM],dtype=complex)
    phi=np.linspace(-sphi,sphi,N)
    theta=np.linspace(-stheta,stheta,M)
    dp=phi[1]-phi[0]
    dt=theta[1]-theta[0]

    GHz=1000000000

    Ecp=base*GHz
    Ect=tpRatio*base*GHz
    El=alpha*GHz
    Ej=beta*GHz

    count=0
    for j in range(M):
        for i in range(N):
            pos=np.array([i,j])
            H[count,C21(pos,size)]= +4*(Ect*ng**2)\
                                    +El*(phi[i]-np.pi*phiE)**2\
                                    -2*Ej*np.cos(phi[i])*np.cos(theta[j])
            H=SDiffPhi(H,pos,-4*Ecp,dp,count,size)
            H=FDiffTheta(H,pos,+1j*8*Ect*ng,dt,count,size)
            H=SDiffTheta(H,pos,-4*Ect,dt,count,size)
            count+=1
    if GetHamiltonian:
        return H
    if Isincludestate:
        a,b=sp.sparse.linalg.eigsh(H,Enum,which='SR')
        return a,b
    a,b=sp.sparse.linalg.eigsh(H,Enum,which='SR')
    return a


In [ ]:
# Job Creator Spot
n1=5
n2=5
n3=5
size=[50,25]
Enum=20

sSweepng=0.5
sSweeppE=0.5

En=np.zeros([n1*n2*n3,Enum+2+1],dtype=complex)
phiE=np.linspace(-sSweeppE,sSweeppE,n1)
ng=np.linspace(-sSweepng,sSweepng,n2)
base=np.linspace(0.5,5,n3)

for i in range(n1):
    for j in range(n2):
        for k in range(n3):
            En[k*n1*n2+j*n1+i,0]=phiE[i]
            En[k*n1*n2+j*n1+i,1]=ng[j]
            En[k*n1*n2+j*n1+i,2]=base[k]
            En[k*n1*n2+j*n1+i,3:]=maingeneral(phiE[i],ng[j],size,stheta=2*np.pi)[:Enum]
# np.save("Ideal-run.nby",En)



    # Experimental
    # base=2.5
    # tpRatio=3.2
    # alpha=1.88
    # beta=27.2




    # # Ideal
    # base=1
    # tpRatio=1
    # alpha=1
    # beta=1/200


In [ ]:
# Run Overnight
# Note ive swap ij
n1=41
n2=41
n3=15
    # base=2.5
    # tpRatio=3.08
    # alpha=1.88
    # beta=27.2

size=[40,20]
N,M=size
Enum=7
En=np.zeros([n1*n2,Enum+2],dtype=complex)
sSweepng=0.5
sSweeppE=0.5
phiE=np.linspace(-0,sSweeppE,n1)
ng=np.linspace(-0,sSweepng,n2)
wfdat=np.zeros([N*M,N*M,Enum])
# Experiment
for i in range(n1):
    for j in range(n2):
        num=j*n1+i
        En[num,0]=phiE[i]
        En[num,1]=ng[j]
        temp,temp1=maingeneral(phiE[i],ng[j],size,stheta=np.pi,base=2.5,tpRatio=3.08,alpha=1.88,beta=27.2,Isincludestate=True)
        En[num,2:]=temp[:Enum]
        wfdat[num,:,:]=temp1[:,:Enum]
        print("Solved: ",num)
np.save("another-Experiment-result",En)
np.save("another-Experiment-result-wf",wfdat)

En=np.zeros([n1*n2,Enum+2],dtype=complex)
for i in range(n1):
    for j in range(n2):
        num=j*n1+i
        En[num,0]=phiE[i]
        En[num,1]=ng[j]
        temp,temp1=maingeneral(phiE[i],ng[j],size,stheta=np.pi,base=2.5,tpRatio=3.08,alpha=1.88,beta=27.2,Isincludestate=True)
        En[num,2:]=temp[:Enum]
        wfdat[num,:,:]=temp1[:,:Enum]
        print("Solved: ",num)
np.save("another-Ideal-result",En)
np.save("another-Ideal-result-wf",wfdat)
# size=[40,20]
# n1=7
# n2=7
# n3=15
# # overshoot=0.

# phiE=np.linspace(0,sSweeppE,n1)
# ng=np.linspace(0,sSweepng,n2)
# base=np.linspace(0.2,5,n3)
# tpRatio=np.linspace(1,5,n3)
# alpha=np.linspace(0.2,10,n3)
# beta=np.linspace(1/200,27.2,n3)
# En=np.zeros([n1*n2*n3,Enum+2+1],dtype=complex)
# # Varies base
# for i in range(n1):
#     for j in range(n2):
#         for k in range(n3):
#             En[k*n1*n2+j*n1+i,0]=phiE[i]
#             En[k*n1*n2+j*n1+i,1]=ng[j]
#             En[k*n1*n2+j*n1+i,2]=base[k]
#             En[k*n1*n2+j*n1+i,3:]=maingeneralsph(phiE[i],ng[j],size,stheta=np.pi,base=base[k],Enum=Enum+1)[:Enum]
# np.save("base-varied-result",En)
# # Varies tpRatio
# En=np.zeros([n1*n2*n3,Enum+2+1],dtype=complex)
# for i in range(n1):
#     for j in range(n2):
#         for k in range(n3):
#             En[k*n1*n2+j*n1+i,0]=phiE[i]
#             En[k*n1*n2+j*n1+i,1]=ng[j]
#             En[k*n1*n2+j*n1+i,2]=tpRatio[k]
#             En[k*n1*n2+j*n1+i,3:]=maingeneralsph(phiE[i],ng[j],size,stheta=np.pi,tpRatio=tpRatio[k],Enum=Enum+1)[:Enum]
# np.save("tpRatio-varied-result",En)
# # Varies alpha
# En=np.zeros([n1*n2*n3,Enum+2+1],dtype=complex)
# for i in range(n1):
#     for j in range(n2):
#         for k in range(n3):
#             En[k*n1*n2+j*n1+i,0]=phiE[i]
#             En[k*n1*n2+j*n1+i,1]=ng[j]
#             En[k*n1*n2+j*n1+i,2]=alpha[k]
#             En[k*n1*n2+j*n1+i,3:]=maingeneralsph(phiE[i],ng[j],size,stheta=np.pi,alpha=alpha[k],Enum=Enum+1)[:Enum]
# np.save("alpha-varied-result",En)
# # Varies beta
# En=np.zeros([n1*n2*n3,Enum+2+1],dtype=complex)
# for i in range(n1):
#     for j in range(n2):
#         for k in range(n3):
#             En[k*n1*n2+j*n1+i,0]=phiE[i]
#             En[k*n1*n2+j*n1+i,1]=ng[j]
#             En[k*n1*n2+j*n1+i,2]=beta[k]
#             En[k*n1*n2+j*n1+i,3:]=maingeneralsph(phiE[i],ng[j],size,stheta=np.pi,beta=beta[k],Enum=Enum+1)[:Enum]
# np.save("beta-varied-result",En)

# size=[30,15]
# n1=7
# n2=7
# n3=4

# phiE=np.linspace(-0,sSweeppE,n1)
# ng=np.linspace(-0,sSweepng,n2)
# base=np.linspace(0.2,5,n3)
# tpRatio=np.linspace(1,5,n3)
# alpha=np.linspace(0.2,10,n3)
# beta=np.linspace(1/200,27.2,n3)
# En=np.zeros([n1*n2*n3*n3*n3*n3,Enum+2+4],dtype=complex)
# for i in range(n1):
#     for j in range(n2):
#         for k1 in range(n3):
#             for k2 in range(n3):
#                 for k3 in range(n3):
#                     for k4 in range(n3):
#                         En[k4*n1*n2*n3**3+k3*n1*n2*n3**2+k2*n1*n2*n3+k1*n1*n2+j*n1+i,0]=phiE[i]
#                         En[k4*n1*n2*n3**3+k3*n1*n2*n3**2+k2*n1*n2*n3+k1*n1*n2+j*n1+i,1]=ng[j]
#                         En[k4*n1*n2*n3**3+k3*n1*n2*n3**2+k2*n1*n2*n3+k1*n1*n2+j*n1+i,2]=base[k1]
#                         En[k4*n1*n2*n3**3+k3*n1*n2*n3**2+k2*n1*n2*n3+k1*n1*n2+j*n1+i,3]=tpRatio[k2]
#                         En[k4*n1*n2*n3**3+k3*n1*n2*n3**2+k2*n1*n2*n3+k1*n1*n2+j*n1+i,4]=alpha[k3]
#                         En[k4*n1*n2*n3**3+k3*n1*n2*n3**2+k2*n1*n2*n3+k1*n1*n2+j*n1+i,5]=beta[k4]
#                         En[k4*n1*n2*n3**3+k3*n1*n2*n3**2+k2*n1*n2*n3+k1*n1*n2+j*n1+i,6:]=maingeneralsph(phiE[i],ng[j],size,stheta=np.pi,base=base[k1],tpRatio=tpRatio[k2],alpha=alpha[k3],beta=beta[k4],Enum=Enum+1)[:Enum]
# np.save("all-varied-result",En)



# 10<Ej30 ghz
#  0.2<Ec<assume 10  ghz
# El 0.2-0.3 ghz
    # Experimental
    # base=2.5
    # tpRatio=3.2
    # alpha=1.88
    # beta=27.2

    # # Ideal
    # base=1
    # tpRatio=1
    # alpha=1
    # beta=1/200
# vectorization

In [ ]:
def a():
    return [1,2,3,4,5],[2,2,3,4,5]

# np.array(a()[:,1]).flatten()
b,c=[*maingeneral(phiE[i],ng[j],size,stheta=np.pi,base=2.5,tpRatio=3.08,alpha=1.88,beta=27.2,Isincludestate=True)]
d=b[:4]
e=c[:,:4]


In [ ]:
# Testing Helper Function
import ipywidgets as widgets
size=[5,8]
N,M=size
NM=N*M
H=main(1,1,size,GetHamiltonian=True)
np.set_printoptions(linewidth=1600)
dump=np.zeros([NM,M,N],dtype=complex)
for i in range(NM):
    dump[i]=R12(H[i],size)

x = widgets.IntSlider(description='x',max=N-1)
y = widgets.IntSlider(description='y',max=M-1)
def f(x, y):
    print(dump[C21([x,y],size)])

out = widgets.interactive_output(f, {'x': x, 'y': y},)

widgets.VBox([widgets.HBox([x, y]), out],layout=widgets.Layout(width='1400px'))

In [ ]:
# Test performance
import timeit
from time import perf_counter

num=7
size=[40,20]

init=perf_counter()
maingeneralsph(0.2,0.3,size,Enum=num)
end=perf_counter()
sparseh=end-init

init=perf_counter()
Ee,wf=maingeneral(0.2,0.3,size,Isincludestate=True)
end=perf_counter()
normal=end-init

print("Normal: ",normal)
# print("Normalh: ",normalh)
# print("sparse: ",sparse)
print("sparseh: ",sparseh)

# Somehow it was faster the normal one on my laptop

init=perf_counter()
maingeneral(0.5,0.5,size)
end=perf_counter()
normal=end-init

init=perf_counter()
maingeneralsph(0.5,0.5,size,Enum=num)
end=perf_counter()
sparseh=end-init

print("Normal: ",normal)
# print("Normalh: ",normalh)
# print("sparse: ",sparse)
print("sparseh: ",sparseh)
Ee